<a href="https://colab.research.google.com/github/AswiniNaresh/MachineLearning/blob/main/Autosys1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load the data
job_exec_log = pd.read_csv('/content/drive/My Drive/JobExecChgLog.csv')
master_config = pd.read_csv('/content/drive/My Drive/MasterJobConfig.csv')


In [ ]:
# Data Preprocessing
def preprocess_data(job_exec_log, master_config):
    # Convert datetime columns
    datetime_columns = ['StartTime', 'EndTime', 'InsertDtTm', 'DataDt', 'EffDt']
    for col in datetime_columns:
        job_exec_log[col] = pd.to_datetime(job_exec_log[col])

    # Merge datasets on appropriate columns (JobName and JobType)
    merged_data = pd.merge(
        job_exec_log,
        master_config,
        left_on=['JobName', 'JobType'],
        right_on=['Autosys_JobName', 'Autosys_JobType'],
        how='left'
    )

    # Filter for successful jobs (assuming 'Status' column indicates success)
    merged_data = merged_data[merged_data['Status'] == 'SUCCESS']

    # Create the 'CompleteTime' (EndTime - StartTime)
    merged_data['CompleteTime'] = (merged_data['EndTime'] - merged_data['StartTime']).dt.total_seconds()

    # Drop rows where 'Duration' or 'CompleteTime' is missing
    merged_data = merged_data.dropna(subset=['Duration', 'CompleteTime'])

    return merged_data


In [ ]:
# Feature Engineering
def create_features(df):
    # Create a copy to avoid modifications to original dataframe
    df = df.copy()

    # Time-based features
    df['hour_of_day'] = df['StartTime'].dt.hour
    df['day_of_week'] = df['StartTime'].dt.dayofweek
    df['month'] = df['StartTime'].dt.month
    df['is_weekend'] = df['StartTime'].dt.dayofweek.isin([5, 6]).astype(int)

    # Calculate historical statistics for each job
    agg_functions = {
        'Duration': ['mean', 'std', 'min', 'max', 'count']
    }

    historical_stats = df.groupby('JobName').agg(agg_functions)
    historical_stats.columns = [
        'hist_mean_duration',
        'hist_std_duration',
        'hist_min_duration',
        'hist_max_duration',
        'hist_count'
    ]
    historical_stats = historical_stats.reset_index()

    # Merge historical stats back to main dataframe
    df = pd.merge(df, historical_stats, on='JobName', how='left')

    # Calculate recent statistics (last 7 days)
    df['StartDate'] = df['StartTime'].dt.date
    for job in df['JobName'].unique():
        job_data = df[df['JobName'] == job].copy()
        job_data = job_data.sort_values('StartTime')

        # Calculate 7-day rolling statistics
        job_data['rolling_7d_mean'] = job_data['Duration'].rolling(
            window=7, min_periods=1).mean()
        job_data['rolling_7d_std'] = job_data['Duration'].rolling(
            window=7, min_periods=1).std()

        # Update the main dataframe
        df.loc[df['JobName'] == job, 'rolling_7d_mean'] = job_data['rolling_7d_mean']
        df.loc[df['JobName'] == job, 'rolling_7d_std'] = job_data['rolling_7d_std']

    # Fill NaN values with historical statistics
    df['rolling_7d_mean'] = df['rolling_7d_mean'].fillna(df['hist_mean_duration'])
    df['rolling_7d_std'] = df['rolling_7d_std'].fillna(df['hist_std_duration'])

    # Job complexity features
    df['parameters_count'] = df['ParameterTxt'].str.count(',').fillna(0) + 1
    df['has_dependencies'] = (~df['DepJobList'].isna()).astype(int)

    # Drop temporary columns
    df = df.drop(['StartDate'], axis=1)

    return df

In [ ]:
# Model Training
def train_model(df):
    # Prepare features
    feature_columns = [
        'hour_of_day', 'day_of_week', 'month', 'is_weekend',
        'hist_mean_duration', 'hist_std_duration',
        'hist_min_duration', 'hist_max_duration',
        'rolling_7d_mean', 'rolling_7d_std',
        'parameters_count', 'has_dependencies'
    ]

    # Encode categorical variables
    categorical_columns = ['JobName', 'JobType', 'AsGroup', 'Machine']
    encoders = {}

    for col in categorical_columns:
        le = LabelEncoder()
        df[f'{col}_encoded'] = le.fit_transform(df[col])
        encoders[col] = le
        feature_columns.append(f'{col}_encoded')

    # Prepare data for training
    X = df[feature_columns]
    y = df['CompleteTime']  # Target variable is CompleteTime (in seconds)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Define model and parameters for grid search
    model = RandomForestRegressor(random_state=42)
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }

    # Perform grid search
    grid_search = GridSearchCV(
        model, param_grid, cv=5,
        scoring='neg_mean_squared_error', n_jobs=-1
    )
    grid_search.fit(X_train, y_train)

    # Get best model
    best_model = grid_search.best_estimator_

    # Evaluate model
    y_pred = best_model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("\nModel Performance:")
    print(f"Mean Absolute Error: {mae:.2f} seconds")
    print(f"Root Mean Squared Error: {rmse:.2f} seconds")
    print(f"R² Score: {r2:.3f}")

    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)

    print("\nTop 10 Most Important Features:")
    print(feature_importance.head(10))

    return best_model, encoders, feature_columns

print("Starting data preprocessing...")
merged_data = preprocess_data(job_exec_log, master_config)

print("Creating features...")
featured_data = create_features(merged_data)

print("Training model...")
model, encoders, feature_columns = train_model(featured_data)

# Save model and related objects
save_path = '/content/drive/My Drive/'
!mkdir -p {save_path}

joblib.dump(model, f'{save_path}model.joblib')
joblib.dump(encoders, f'{save_path}encoders.joblib')
joblib.dump(feature_columns, f'{save_path}feature_columns.joblib')

print("\nModel and related objects saved successfully!")

Starting data preprocessing...
Creating features...
Training model...

Model Performance:
Mean Absolute Error: 216.00 seconds
Root Mean Squared Error: 287.85 seconds
R² Score: 0.130

Top 10 Most Important Features:
               feature  importance
8      rolling_7d_mean    0.360740
9       rolling_7d_std    0.209576
0          hour_of_day    0.101423
1          day_of_week    0.049322
4   hist_mean_duration    0.041470
12     JobName_encoded    0.038662
5    hist_std_duration    0.034676
6    hist_min_duration    0.032615
7    hist_max_duration    0.032152
13     JobType_encoded    0.029164
mkdir: cannot create directory ‘/content/drive/My’: Operation not supported

Model and related objects saved successfully!
